# MiniProyecto Deep Learning
## Elaborado por Fabián Calvo Castillo - Florencia Pellegrini

In [ ]:
#%pip install feedparser trafilatura
#%pip install google-generativeai

In [ ]:
#%pip install python-dotenv

In [ ]:
#%pip install edge-tts
#%pip install nest_asyncio
#%pip install openai-whisper

In [ ]:
#!pip install moviepy==1.0.3
#!pip install Pillow

In [1]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)
from news_fetcher import get_all_news
import google.generativeai as genai
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin
import t2s
importlib.reload(t2s)
from t2s import generar_audio

C:\Users\fcalv\AppData\Local\Temp\ipykernel_11972\2596570605.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Paso 1: Obtener noticias

In [2]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)

from news_fetcher import get_all_news
articulos = get_all_news(num_per_source=3)

RECOPILANDO NOTICIAS
\n-> Leyendo RSS de todas las fuentes...
  [ABC] 0 articulos en RSS
  [ElDiario] 3 articulos en RSS
  [Europa Press] 3 articulos en RSS
  [La Vanguardia] 3 articulos en RSS
  inutos] 3 articulos en RSS
  [Antena 3] 3 articulos en RSS
  [RTVE] 3 articulos en RSS
  [El Pais] 3 articulos en RSS
  [El Mundo] 3 articulos en RSS
\n-> Extrayendo texto completo (24 articulos)...
  Procesando: Trump ordena “posponer” los ataques contra plantas ener...
    -> Texto completo (10521 chars)
  Procesando: Podemos se abstendrá el jueves en la votación del decre...
    -> Texto completo (3363 chars)
  Procesando: El PP pide al Gobierno por carta nuevas medidas por la ...
    -> Texto completo (3553 chars)
  Procesando: Trump ordena posponer cinco días los ataques a centrale...
    -> Texto completo (4913 chars)
  Procesando: Guerra en Irán | Directo: Irán niega negociaciones con ...
    -> Texto completo (2435 chars)
  Procesando: El Ibex 35 mantiene su repunte por encima del 1% a

Con esta función se guarda una variable articulos, que es una lista de diccionarios Python, uno por cada artículo. 
Así luego el LLM recibirá todos los textos completos agrupados por tema y devolverá un único resumen redactado de forma neutral, sin subjetivismo político. 

In [3]:
# Comprobación

for i, art in enumerate(articulos):
    print(f"\n{'='*40}")
    print(f"Artículo {i+1}")
    print(f"Fuente:  {art['fuente']}")
    print(f"Título:  {art['titulo']}")
    print(f"Origen:  {art['texto_origen']}")
    print(f"Chars:   {len(art['texto_completo'])}")
    print(f"Preview: {art['texto_completo'][:200]}...")


Artículo 1
Fuente:  ElDiario
Título:  Trump ordena “posponer” los ataques contra plantas energéticas de Irán tras unas “conversaciones buenas” que Teherán niega
Origen:  completo
Chars:   10521
Preview: Trump ordena “posponer” los ataques contra plantas energéticas de Irán tras unas “conversaciones buenas” que Teherán niega
¿Se abre una nueva fase en la guerra de EEUU e Israel en Irán o es otro banda...

Artículo 2
Fuente:  ElDiario
Título:  Podemos se abstendrá el jueves en la votación del decreto por la guerra de Irán y deja al Gobierno en manos de Junts y PP
Origen:  completo
Chars:   3363
Preview: Podemos se abstendrá el jueves en la votación del decreto por la guerra de Irán y deja al Gobierno en manos de Junts y PP
137
PP y Junts serán quienes decidan si se convalida el decreto de medidas fis...

Artículo 3
Fuente:  ElDiario
Título:  El PP pide al Gobierno por carta nuevas medidas por la guerra de Irán para no desvelar si tumbará el decreto
Origen:  completo
Chars:   3553
Previe

## Paso 2: Generación del resumen

Se toma el diccionario generado con las noticias que se recopilaron en distintos medios, y se genera el resumen haciendo uso de la API de Gemini

In [ ]:
import google.generativeai as genai
import importlib
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

genai.configure(api_key=GEMINI_API_KEY)
mi_modelo_gemini = genai.GenerativeModel("gemini-2.5-flash")

# Pipeline de resumen
boletin_final, resumenes = resumir_noticias(articulos, mi_modelo_gemini)

if boletin_final:
    print("\n" + "*"*50)
    print("BOLETÍN DE NOTICIAS")
    print("*"*50 + "\n")
    print(boletin_final)
    guardar_boletin(boletin_final) 


RESUMIENDO 24 ARTÍCULOS

-> Agrupando artículos por temas...
   4 temas identificados:
   - EEUU e Irán: Negociaciones, Ultimátum y Negación (11 fuentes: ElDiario, Europa Press, Europa Press, La Vanguardia, La Vanguardia, 20minutos, 20minutos, RTVE, El Pais, El Pais, El Mundo)
   - Impacto Económico del Conflicto de Irán (2 fuentes: La Vanguardia, El Pais)
   - Política Española ante el Conflicto de Irán (2 fuentes: ElDiario, ElDiario)
   - Reacciones y Diplomacia Internacional (4 fuentes: 20minutos, Antena 3, El Mundo, El Mundo)

-> Generando resúmenes por tema...
   [1/4] EEUU e Irán: Negociaciones, Ultimátum y Negación (11 fuentes)...
      OK (1110 chars)
   [2/4] Impacto Económico del Conflicto de Irán (2 fuentes)...
      OK (961 chars)
   [3/4] Política Española ante el Conflicto de Irán (2 fuentes)...
      OK (726 chars)
   [4/4] Reacciones y Diplomacia Internacional (4 fuentes)...
      OK (1049 chars)

-> Montando boletín final...
   Error montando boletín, concatenando dir

## Paso 3: Síntesis de voz (T2S)

In [5]:
import importlib
import t2s
importlib.reload(t2s)
from t2s import generar_audio

ruta_audio, ruta_subtitulos = generar_audio(boletin_final)

Generando audio...
  Voz:         es-ES-AlvaroNeural
  Chars:       4485
  Audio:       audios/audio_2026-03-23_20-24-16.mp3
Audio generado! (1.55 MB)
  Generando subtítulos con Whisper...


c:\Users\fcalv\anaconda3\envs\MCD_25_26\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


FileNotFoundError: [WinError 2] El sistema no puede encontrar el archivo especificado

In [ ]:
# Verificar que el VTT tiene contenido
with open(ruta_subtitulos, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"VTT tiene {len(contenido)} chars")
print(contenido[:300])

VTT tiene 11449 chars
WEBVTT

00:00:00.000 --> 00:00:01.760
Conflicto y negociaciones entre

00:00:01.760 --> 00:00:02.980
Estados Unidos y Irán.

00:00:04.019 --> 00:00:05.099
El presidente de Estados

00:00:05.099 --> 00:00:07.400
Unidos, Donald Trump, anunció

00:00:07.400 --> 00:00:08.320
la orden de posponer

00:00:


## Paso 4: Generación del video resumen 

In [ ]:
import subprocess
resultado = subprocess.run(["where", "magick"], capture_output=True, text=True)
print(resultado.stdout)

C:\Program Files\ImageMagick-7.1.2-Q16-HDRI\magick.exe



In [ ]:
import importlib
import video_maker
importlib.reload(video_maker)
from video_maker import generar_video

PEXELS_API_KEY = os.getenv('PEXELS_API_KEY')

ruta_video = generar_video(ruta_audio, ruta_subtitulos, resumenes, PEXELS_API_KEY)


GENERANDO VÍDEO

1. Cargando audio...
   Duración total:    290.5s
   Temas:             6
   Duración por tema: 48.4s

2. Buscando imágenes en Pexels...
  Tema: Conflicto y Negociaciones entre EEUU e Irán...
    Keywords: 'conflicto negociaciones entre eeuu'
    Imagen guardada: imagenes/img_8842911119247063826.jpg
  Tema: Crímenes y Sucesos en España...
    Keywords: 'crímenes sucesos españa'
    Imagen guardada: imagenes/img_8572434273152678577.jpg
  Tema: Noticias Regionales y Locales de España...
    Keywords: 'noticias regionales locales españa'
    Imagen guardada: imagenes/img_7089684556216950599.jpg
  Tema: Crisis Energética Global...
    Keywords: 'crisis energética global'
    Imagen guardada: imagenes/img_251657473663213057.jpg
  Tema: Geopolítica: Preparación Europea...
    Keywords: 'geopolítica preparación europea'
    Imagen guardada: imagenes/img_7432720517247729759.jpg
  Tema: Deportes: Atletismo Español...
    Keywords: 'deportes atletismo español'
    Imagen guarda